### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

c:\Users\matcg\OneDrive\Documents\GitHub\LangChain_Updated\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E91C0357F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E91C036270>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie out of 10', 'type': 'number'}}, 'required': ['title', 'yea

In [4]:
model.invoke("Provide me details of the movie Inception")

AIMessage(content='<think>\nOkay, the user is asking for details about the movie Inception. Let me start by recalling what I know about it. Inception is a 2010 film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, and it\'s a sci-fi thriller. The plot involves dreams and heists, right? The main character is Dom Cobb, a thief who steals information by entering the subconscious. The concept of planting an idea is called "inception," which is the opposite of extraction. \n\nThe user might want a basic summary, but maybe also some deeper elements. Let me check the main characters. There\'s Arthur (played by Joseph Gordon-Levitt), Eames (Tom Hardy), and Ariadne (Ellen Page). Each has specific roles in the dream-sharing process. The plot has multiple layers with different dream levels, which can be confusing, so the user might appreciate a clear breakdown. \n\nThe music is by Hans Zimmer, and the music is a significant part of the movie, especially the use of the slowed-do

In [8]:
response = model_with_structure.invoke("Provide me details of the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide me details of the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. First, I need to recall the information about Inception. The title is obviously "Inception". The director is Christopher Nolan. The release year was 2010. As for the rating, I think it\'s around 8.8 on IMDb, but since the user didn\'t specify the source, I\'ll just use a general rating. I need to make sure all required parameters are included. Let me check the required fields again: title, year, director, rating. Yep, I have all of those. So I\'ll structure the tool call with those parameters. I should format the JSON correctly, making sure the types are right—year is an integer, rating is a number. Alright, that should do it.\n', 'tool_calls': [{'id': 'ps0m4xqsw', 'function': {'arguments': '{"director":"Christopher Nolan","

### Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    cast: list[Actor] = Field(..., description="A list of actors in the movie")
    genres: list[str] = Field(..., description="The genres of the movie")
    budget: float | None = Field(..., description="The budget of the movie in millions USD, if available")

In [14]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide me details of the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Joker / Bane / Caelus')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

## TypeDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [22]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Provide me details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [24]:
class Actor(TypedDict):
    name: str
    role: str 

class MovieDetails(TypedDict):
    title: str 
    year: int 
    cast: list[Actor] 
    genres: list[str] 
    budget: float | None = Field(..., description="The budget of the movie in millions USD, if available")

In [26]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide me details of the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers Assemble',
 'year': 2012}

In [28]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [29]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='dd8d355b-2241-47ee-9bae-1e04b76faa98'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 933, 'prompt_tokens': 204, 'total_tokens': 1137, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DI0wb6e3q7OYyeZ1F9E0s2dCXdhDW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cda1f-5486-7e00-be46-421f5ca794f1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'outpu

In [32]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [33]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [34]:
## DataClasses
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')